Legacy mount setup completed, but execution failed with the expected is not whitelisted exception. This happens because the shared academy cluster enforces Unity Catalog in Shared Access Mode, which explicitly blocks legacy DBFS mounts for security reasons. This perfectly demonstrates why legacy mounts are deprecated and why our new UC External Locations are the modern standard.

In [0]:
storage_account_name = "dlspl21databricks"
container_name = "artemzharkov10"
mount_point = f"/mnt/{container_name}_legacy"

try:
    client_id = dbutils.secrets.get(scope="default2", key="sp-databricks-adls-appid")
    client_secret = dbutils.secrets.get(scope="default2", key="sp-databricks-adls-appkey")
    tenant_id = dbutils.secrets.get(scope="default2", key="tenant-id")

    # configurations for the mount
    configs = {
        "fs.azure.account.auth.type": "OAuth",
        "fs.azure.account.oauth.provider.type": "org.apache.hadoop.fs.azurebfs.oauth2.ClientCredsTokenProvider",
        "fs.azure.account.oauth2.client.id": client_id,
        "fs.azure.account.oauth2.client.secret": client_secret,
        "fs.azure.account.oauth2.client.endpoint": f"https://login.microsoftonline.com/{tenant_id}/oauth2/token"
    }

    dbutils.fs.mount(
        source = f"abfss://{container_name}@{storage_account_name}.dfs.core.windows.net/",
        mount_point = mount_point,
        extra_configs = configs
    )
    print("Mount successful!")

    test_path = f"dbfs:{mount_point}/raw_data/finance_history/GOLD.csv" 
    df = spark.read.csv(test_path, header=True, inferSchema=True)
    display(df.limit(5))

except Exception as e:
    print(f"Error exception: {e}")